In [5]:
"""
Read data from the apache-logs.txt file
Load and display the data
"""

file_df = spark.read.format("text").load(path = "/workspaces/pyspark_udemy_codespace/data/apache-logs.txt")
file_df.show() # each line has become one row of a single column

+--------------------+
|               value|
+--------------------+
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
|83.149.9.216 - - ...|
+--------------------+
only showing top 20 rows


In [12]:
"""
Develop an strategy to extract the following fields
    ip_address: It is the IP address of the site visitor.
    visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
    visit_resource: Which resource from our website was accessed
    referring_url: It is the clean URL of the referring website.
"""

# develop a regex
log_reg = r'^(\S+) (\S+) (\S+) \[([\w:/]+\s[+\-]\d{4})\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+) "(\S+)" "([^"]*)'

# apply regex to parse record
from pyspark.sql.functions import regexp_extract

logs_df = (
    file_df.select(
        regexp_extract("value", log_reg, 1).alias("ip_address"),
        regexp_extract("value", log_reg, 4).alias("visit_timestamp"),
        regexp_extract("value", log_reg, 6).alias("visit_resource"),
        regexp_extract("value", log_reg, 10).alias("referring_url")
    )
)
# logs_df.show()

# refine results with other transformations
from pyspark.sql.functions import to_timestamp, col, substring_index
logs_refined_df = logs_df.withColumns({
    "visit_timestamp": to_timestamp("visit_timestamp", "dd/MMM/yyyy:HH:mm:ss Z"),
    "referring_url": substring_index(col("referring_url"), '/', 3)
})
logs_refined_df.show()

+------------+-------------------+--------------------+--------------------+
|  ip_address|    visit_timestamp|      visit_resource|       referring_url|
+------------+-------------------+--------------------+--------------------+
|83.149.9.216|2015-05-17 10:05:03|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:43|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:47|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:12|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:07|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:34|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:57|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:50|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:24|/presentations/lo...|http://semicomple...|
|83.149.9.216|2015-05-17 10:05:50|/presentations/lo...|http://semicomple...|